## 1. Setup

Load all dependencies, connect to MySQL, and pull `nashville_housing_clean` into a DataFrame. The `connect_args` pattern is used for the SQLAlchemy connection — credentials are passed as a dictionary, not embedded in the URL string, because the password contains characters (`$`, `@`) that SQLAlchemy's URL parser would misread as delimiters.

In [ ]:
import os
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats
from sqlalchemy import create_engine, text
from dotenv import load_dotenv

# Load environment variables from .env in the project root.
# interpolate=False prevents the dotenv parser from treating $ as a
# variable expansion character — relevant because the MySQL password
# contains $ characters.
load_dotenv("../.env", interpolate=False)

DB_HOST     = os.getenv("DB_HOST")
DB_PORT     = os.getenv("DB_PORT", "3306")
DB_NAME     = os.getenv("DB_NAME")
DB_USER     = os.getenv("DB_USER")
DB_PASSWORD = os.getenv("DB_PASSWORD")
BLS_API_KEY = os.getenv("BLS_API_KEY")

# SQLAlchemy engine — credentials passed via connect_args, not the URL,
# to bypass URL parsing entirely and treat all characters as literals.
engine = create_engine(
    f"mysql+pymysql://{DB_USER}@{DB_HOST}:{DB_PORT}/{DB_NAME}",
    connect_args={"password": DB_PASSWORD}
)

# Verify connection.
with engine.connect() as conn:
    result = conn.execute(text("SELECT COUNT(*) FROM nashville_housing_clean")).fetchone()
    print(f"Connected. nashville_housing_clean row count: {result[0]:,}") # pyright: ignore[reportOptionalSubscript]

### Load the clean table

In [ ]:
df = pd.read_sql("SELECT * FROM nashville_housing_clean", engine)

print(f"Shape: {df.shape}")
print(f"\nColumn dtypes:\n{df.dtypes}")
print(f"\nSample rows:")
df.head(3)

## 2. CPI-U Ingestion

The US Bureau of Labor Statistics provides CPI data through a free public API. We fetch the annual average CPI-U for 2013–2016, the period covered by the dataset.

**Why CPI-U?** CPI-U (Consumer Price Index for All Urban Consumers) is the standard measure used for general inflation adjustment. It is the same series used by the Federal Reserve and most economic research as a broad cost-of-living measure.

**Why the BLS API instead of a CSV?** A hardcoded CSV would need manual updates if the dataset period ever changed. The API call is reproducible, auditable, and works on any machine with an internet connection.

**Series ID used:** `CUUR0000SA0` — this is the national CPI-U, seasonally unadjusted, all items. Seasonally unadjusted is correct here because we are comparing annual averages, not month-to-month movements.

In [ ]:
# BLS API v2 endpoint — requires registration for higher rate limits (500/day vs 25/day).
# The API key is loaded from .env — never hardcoded.
BLS_URL = "https://api.bls.gov/publicAPI/v2/timeseries/data/"

payload = {
    "seriesid": ["CUUR0000SA0"],   # National CPI-U, all items, not seasonally adjusted
    "startyear": "2013",
    "endyear": "2016",
    "annualaverage": True,          # Request the annual average alongside monthly values
    "registrationkey": BLS_API_KEY
}

response = requests.post(BLS_URL, json=payload)
response.raise_for_status()

data = response.json()

if data["status"] != "REQUEST_SUCCEEDED":
    raise ValueError(f"BLS API request failed: {data['message']}")

print("BLS API request succeeded.")
print(f"Series fetched: {[s['seriesID'] for s in data['Results']['series']]}")

### Parse CPI response into a usable DataFrame

The BLS API returns monthly values and, if requested, an annual average labelled with period code `M13`. We extract only the annual average rows because we are adjusting by year, not by month — monthly adjustment would require matching on sale month, which adds complexity without meaningful precision gain for this analysis.

In [ ]:
series_data = data["Results"]["series"][0]["data"]

# M13 is the BLS code for annual average.
annual = [
    {"year": int(row["year"]), "cpi": float(row["value"])}
    for row in series_data
    if row["period"] == "M13"
]

cpi_df = pd.DataFrame(annual).sort_values("year").reset_index(drop=True)

print("Annual average CPI-U values fetched:")
print(cpi_df.to_string(index=False))

## 3. CPI Adjustment

**Base year: 2013.** This is the earliest year in the dataset. All sale prices are converted to 2013 real dollars using the standard inflation adjustment formula:

> real_price = nominal_price × (CPI_base_year / CPI_sale_year)

A property that sold for $200,000 in 2016, when inflation had raised the CPI by roughly 3% since 2013, would have a real price of approximately $194,000 in 2013 dollars. This adjustment ensures that price comparisons across years reflect genuine value changes, not just the effects of general inflation.

In [ ]:
# Extract the 2013 base CPI value.
cpi_base = cpi_df.loc[cpi_df["year"] == 2013, "cpi"].values[0]
print(f"CPI base year (2013): {cpi_base}")

# Build a year → CPI lookup dictionary.
cpi_lookup = dict(zip(cpi_df["year"], cpi_df["cpi"]))
print(f"\nCPI lookup: {cpi_lookup}")

In [ ]:
# Ensure sale_date is parsed as datetime so we can extract the year.
df["sale_date"] = pd.to_datetime(df["sale_date"], errors="coerce")
df["sale_year"] = df["sale_date"].dt.year

# Check that all years in the data map to years we have CPI data for.
years_in_data = sorted(df["sale_year"].dropna().unique().astype(int).tolist())
years_in_cpi  = list(cpi_lookup.keys())

print(f"Years in sales data:  {years_in_data}")
print(f"Years in CPI lookup:  {years_in_cpi}")

missing = set(years_in_data) - set(years_in_cpi)
if missing:
    print(f"\nWARNING: No CPI data for years: {missing}. These rows will produce null real prices.")
else:
    print("\nAll sale years have CPI coverage. No gaps.")

In [ ]:
# Map CPI values onto each row by sale year and calculate real price.
df["cpi_sale_year"] = df["sale_year"].map(cpi_lookup)
df["sale_price_real"] = df["sale_price"] * (cpi_base / df["cpi_sale_year"])

# Verify: rows where real price is null should only be rows where sale_price or
# sale_year was already null.
null_real = df["sale_price_real"].isna().sum()
null_nominal = df["sale_price"].isna().sum()
print(f"Null nominal prices:  {null_nominal:,}")
print(f"Null real prices:     {null_real:,}")

if null_real > null_nominal:
    print("WARNING: CPI adjustment introduced additional nulls — check cpi_lookup coverage.")
else:
    print("CPI adjustment complete. No new nulls introduced.")

print(f"\nSample comparison (first 5 rows):")
df[["sale_date", "sale_year", "sale_price", "cpi_sale_year", "sale_price_real"]].head()

## 4. Exploratory Analysis — Real Price Distributions

All price analysis from this point uses `sale_price_real` (2013 dollars). Comparisons are meaningful because the inflation component has been removed.

### 4.1 Median real sale price by year

We use the median rather than the mean because sale price distributions are right-skewed — a small number of very high-value properties pull the mean upward and away from the typical transaction. The median is a more honest representation of what a typical buyer paid.

In [ ]:
yearly = (
    df.groupby("sale_year")["sale_price_real"]
    .agg(median_real_price="median", mean_real_price="mean", count="count")
    .reset_index()
)

print(yearly.to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

ax.bar(yearly["sale_year"], yearly["median_real_price"] / 1000,
       color="#2c6fad", width=0.5)

ax.set_xlabel("Sale Year")
ax.set_ylabel("Median Real Sale Price (2013 $000s)")
ax.set_title("Median Real Sale Price by Year\n(CPI-adjusted to 2013 dollars)")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x:,.0f}k"))
ax.set_xticks(yearly["sale_year"])

plt.tight_layout()
plt.savefig("../reports/fig01_median_real_price_by_year.png", dpi=150)
plt.show()
print("Saved: reports/fig01_median_real_price_by_year.png")

### 4.2 Real price by land use category

Land use tells us what the property is — single family, condo, commercial, etc. Understanding how median prices differ by land use tells us whether the market is pricing fundamentally different asset types distinctly, which is a basic reasonableness check on the data before modelling.

In [ ]:
land_use_summary = (
    df.groupby("land_use")["sale_price_real"]
    .agg(median_price="median", count="count")
    .reset_index()
    .sort_values("count", ascending=False)
)

# Show the top 15 categories by volume.
print("Top 15 land use categories by transaction volume:")
print(land_use_summary.head(15).to_string(index=False))

In [ ]:
top_land = land_use_summary.head(12).sort_values("median_price", ascending=True)

fig, ax = plt.subplots(figsize=(10, 6))

ax.barh(top_land["land_use"], top_land["median_price"] / 1000, color="#2c6fad")
ax.set_xlabel("Median Real Sale Price (2013 $000s)")
ax.set_title("Median Real Sale Price by Land Use (Top 12 Categories by Volume)\n(CPI-adjusted to 2013 dollars)")
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x:,.0f}k"))

plt.tight_layout()
plt.savefig("../reports/fig02_median_real_price_by_land_use.png", dpi=150)
plt.show()
print("Saved: reports/fig02_median_real_price_by_land_use.png")

### 4.3 Real price by tax district

Tax districts in the Nashville dataset broadly correspond to geographic sub-markets. Comparing median real prices across districts gives a spatial price map without requiring any geospatial joins at this stage.

In [ ]:
district_summary = (
    df.groupby("tax_district")["sale_price_real"]
    .agg(median_price="median", count="count")
    .reset_index()
    .sort_values("median_price", ascending=False)
)

print("Median real sale price by tax district:")
print(district_summary.to_string(index=False))

In [ ]:
district_sorted = district_summary.sort_values("median_price", ascending=True)

fig, ax = plt.subplots(figsize=(10, 5))

ax.barh(district_sorted["tax_district"], district_sorted["median_price"] / 1000,
        color="#2c6fad")
ax.set_xlabel("Median Real Sale Price (2013 $000s)")
ax.set_title("Median Real Sale Price by Tax District\n(CPI-adjusted to 2013 dollars)")
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x:,.0f}k"))

plt.tight_layout()
plt.savefig("../reports/fig03_median_real_price_by_district.png", dpi=150)
plt.show()
print("Saved: reports/fig03_median_real_price_by_district.png")

### 4.4 Real price growth: which districts grew in real terms?

This is the core payoff of the CPI adjustment. Nominal prices almost always rise over time just because of inflation. The question that matters is: which districts grew faster than inflation (real appreciation), and which merely kept pace or fell behind?

We calculate the percentage change in median real price from 2013 to 2016 for each district with sufficient data in both years.

In [ ]:
# Pivot to get median real price by district and year.
pivot = (
    df.groupby(["tax_district", "sale_year"])["sale_price_real"]
    .median()
    .unstack("sale_year")
    .dropna(subset=[2013, 2016])   # Only districts with data in both anchor years
    .reset_index()
)

pivot.columns.name = None
pivot.columns = ["tax_district"] + [str(int(c)) if isinstance(c, float) else str(c)
                                     for c in pivot.columns[1:]]

# Calculate real growth rate 2013 to 2016.
pivot["real_growth_pct"] = ((pivot["2016"] - pivot["2013"]) / pivot["2013"]) * 100
pivot_sorted = pivot.sort_values("real_growth_pct", ascending=False)

print("Real price growth by tax district (2013 to 2016):")
print(pivot_sorted[["tax_district", "2013", "2016", "real_growth_pct"]].to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

colors = ["#2c6fad" if v >= 0 else "#c0392b" for v in pivot_sorted["real_growth_pct"]]
ax.barh(pivot_sorted["tax_district"], pivot_sorted["real_growth_pct"], color=colors)
ax.axvline(0, color="black", linewidth=0.8, linestyle="--")
ax.set_xlabel("Real Price Change 2013–2016 (%)")
ax.set_title("Real Sale Price Growth by Tax District (2013–2016)\nCPI-adjusted — positive bars = genuine appreciation above inflation")

plt.tight_layout()
plt.savefig("../reports/fig04_real_price_growth_by_district.png", dpi=150)
plt.show()
print("Saved: reports/fig04_real_price_growth_by_district.png")

## 5. Vacancy Discount Analysis

**Hypothesis:** Properties sold as vacant (`sold_as_vacant = 'Y'`) sell at a lower price than occupied properties.

**Why this matters:** If vacant properties consistently sell at a discount, that discount represents a quantifiable market inefficiency — buyers price in the cost and risk of vacancy. Understanding its size and consistency is useful context for any subsequent valuation model.

**Why Mann-Whitney U, not a t-test:**  
A t-test assumes both groups are normally distributed. Sale price distributions are right-skewed (a small number of very expensive properties stretch the tail). The Mann-Whitney U test makes no distributional assumption — it tests whether values from one group tend to be higher or lower than the other group, which is exactly the question we are asking. It is the correct test for skewed continuous data with a binary group variable.

In [ ]:
# Confirm the values present in sold_as_vacant.
print("sold_as_vacant value counts:")
print(df["sold_as_vacant"].value_counts())
print(f"\nNull count: {df['sold_as_vacant'].isna().sum():,}")

In [ ]:
vacant     = df.loc[df["sold_as_vacant"] == "Y", "sale_price_real"].dropna()
not_vacant = df.loc[df["sold_as_vacant"] == "N", "sale_price_real"].dropna()

print(f"Vacant properties:      {len(vacant):,} records")
print(f"Non-vacant properties:  {len(not_vacant):,} records")
print(f"\nMedian real price — vacant:      ${vacant.median():,.0f}")
print(f"Median real price — non-vacant:  ${not_vacant.median():,.0f}")

discount_pct = ((not_vacant.median() - vacant.median()) / not_vacant.median()) * 100
print(f"\nRaw median discount (vacant vs non-vacant): {discount_pct:.1f}%")

In [ ]:
stat, p_value = stats.mannwhitneyu(vacant, not_vacant, alternative="less")

print("Mann-Whitney U Test")
print("Null hypothesis: vacant and non-vacant sale prices come from the same distribution")
print("Alternative:     vacant prices tend to be lower than non-vacant")
print(f"\nU statistic: {stat:,.0f}")
print(f"p-value:      {p_value:.6f}")

alpha = 0.05
if p_value < alpha:
    print(f"\nResult: Reject the null hypothesis (p < {alpha}).")
    print("The vacancy discount is statistically significant.")
else:
    print(f"\nResult: Fail to reject the null hypothesis (p >= {alpha}).")
    print("No statistically significant vacancy discount detected.")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

ax.boxplot(
    [not_vacant / 1000, vacant / 1000],
    tick_labels=["Not Vacant (N)", "Sold as Vacant (Y)"],
    patch_artist=True,
    boxprops=dict(facecolor="#d0e4f5", color="#2c6fad"),
    medianprops=dict(color="#c0392b", linewidth=2),
    flierprops=dict(marker=".", markersize=2, alpha=0.3)
)

ax.set_ylabel("Real Sale Price (2013 $000s)")
ax.set_title("Real Sale Price Distribution: Vacant vs Non-Vacant\n(outliers shown as dots)")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x:,.0f}k"))

plt.tight_layout()
plt.savefig("../reports/fig05_vacancy_price_distribution.png", dpi=150)
plt.show()
print("Saved: reports/fig05_vacancy_price_distribution.png")

In [ ]:
# Fig 05 — fixed version with y-axis capped at $800k
# The uncapped version is distorted by a single $53M outlier in the non-vacant group.
# Capping at $800k retains 99%+ of transactions and makes the distributions visible.

fig, ax = plt.subplots(figsize=(8, 5))

ax.boxplot(
    [not_vacant / 1000, vacant / 1000],
    tick_labels=["Not Vacant (N)", "Sold as Vacant (Y)"],
    patch_artist=True,
    boxprops=dict(facecolor="#d0e4f5", color="#2c6fad"),
    medianprops=dict(color="#c0392b", linewidth=2),
    flierprops=dict(marker=".", markersize=2, alpha=0.3)
)

ax.set_ylabel("Real Sale Price (2013 $000s)")
ax.set_title("Real Sale Price Distribution: Vacant vs Non-Vacant\n(y-axis capped at $800k — outliers above this threshold exist but are excluded for readability)")
ax.set_ylim(0, 800)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x:,.0f}k"))

plt.tight_layout()
plt.savefig("../reports/fig05_vacancy_price_distribution.png", dpi=150)
plt.show()
print("Saved: reports/fig05_vacancy_price_distribution.png — y-axis capped at $800k")

## 6. Land Use Pattern Analysis

How did the mix of transactions across land use categories change over the dataset period? A shift in composition matters for interpretation — if the proportion of high-value commercial transactions increased from 2013 to 2016, that would inflate aggregate price statistics even if individual property values were flat.

In [ ]:
# Top 6 land use categories by total volume — focus on the categories that matter.
top6_categories = (
    df["land_use"].value_counts().head(6).index.tolist()
)
print(f"Top 6 land use categories: {top6_categories}")

In [ ]:
land_year = (
    df[df["land_use"].isin(top6_categories)]
    .groupby(["sale_year", "land_use"])
    .size()
    .reset_index(name="count")
)

# Calculate share within each year.
year_totals = land_year.groupby("sale_year")["count"].transform("sum")
land_year["share_pct"] = (land_year["count"] / year_totals) * 100

print(land_year.to_string(index=False))

In [ ]:
palette = sns.color_palette("tab10", n_colors=len(top6_categories))
color_map = dict(zip(top6_categories, palette))

fig, ax = plt.subplots(figsize=(9, 5))

for category in top6_categories:
    subset = land_year[land_year["land_use"] == category]
    ax.plot(subset["sale_year"], subset["share_pct"],
            marker="o", label=category, color=color_map[category])

ax.set_xlabel("Sale Year")
ax.set_ylabel("Share of Annual Transactions (%)")
ax.set_title("Transaction Share by Land Use Category (Top 6)\n2013–2016")
ax.set_xticks([2013, 2014, 2015, 2016])
ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=8)

plt.tight_layout()
plt.savefig("../reports/fig06_land_use_share_over_time.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: reports/fig06_land_use_share_over_time.png")

## 7. Summary of Findings

Run this cell after completing all sections above. Update the findings text to match your actual output values before committing the notebook.

In [ ]:
print("=" * 60)
print("NOTEBOOK 02 — FINDINGS SUMMARY")
print("=" * 60)

print("\n[1] CPI-U data fetched from BLS API — no manual download.")
print(f"    CPI base year (2013): {cpi_base}")
print(f"    CPI values: {cpi_lookup}")

print("\n[2] Median real sale price by year (2013 dollars):")
for _, row in yearly.iterrows():
    print(f"    {int(row['sale_year'])}: ${row['median_real_price']:,.0f}  (n={int(row['count']):,})")

print("\n[3] Vacancy discount test (Mann-Whitney U):")
print(f"    Median real price — non-vacant: ${not_vacant.median():,.0f}")
print(f"    Median real price — vacant:     ${vacant.median():,.0f}")
print(f"    Raw discount:                   {discount_pct:.1f}%")
print(f"    p-value:                        {p_value:.6f}")
result_text = "SIGNIFICANT" if p_value < 0.05 else "NOT SIGNIFICANT"
print(f"    Result:                         {result_text} at alpha = 0.05")

print("\n[4] Land use composition (top 6 categories, 2013–2016):")
print("    Single family dominant but declining: 63.9% → 60.7%")
print("    Residential condo growing:            20.9% → 27.0%")
print("    'VACANT RES LAND' disappears after 2013 — same asset as 'VACANT RESIDENTIAL LAND'.")
print("    Label inconsistency to consolidate before modelling.")

print("\n[5] Real price growth by district (2013–2016, CPI-adjusted):")
print("    City of Oak Hill:          +64.6% (fastest appreciating)")
print("    City of Berry Hill:        +48.6%")
print("    Urban Services District:   +36.5%")
print("    City of Belle Meade:       +6.3%  (most expensive, slowest growth)")
print("=" * 60)